# Tech Challenge - Fase 1: Análise de Expansão e Crescimento da Olist 
**(Grupo 58)**

## 1. Contexto e Pergunta de Negócio

**Pergunta Principal:** Onde estão as maiores oportunidades de expansão de receita da Olist, considerando o ticket médio e o impacto do frete?

**Justificativa Estratégica:**
No cenário do e-commerce brasileiro, o crescimento financeiro está diretamente atrelado à eficiência logística. Segundo um [levantamento publicado pela ABIACOM](https://abiacom.org/noticias/plataforma-de-intermediacao-de-fretes-e-ferramenta-importante-nas-vendas-online/), o frete é responsável por quase 66% dos custos logísticos de uma loja virtual, e o alto valor da entrega leva a 80% dos abandonos de carrinho. 

Como o modelo de negócios da Olist atua diretamente na intermediação e otimização desses contratos logísticos, através de soluções como o [Envios da Olist, que conecta pequenos e médios lojistas a transportadoras parceiras para otimizar custos](https://ajuda.olist.com/integracoes-e-automacao/envios-da-olist-%E2%80%94-guia-completo-de-integracao-e-uso), cruzar os dados de **Faturamento (Receita)** e **Frete (Custo)** me permite analisar onde estão as melhores oportunidades de expansão de mercado e onde o custo de envio está atrapalhando as vendas.

In [5]:
# Manipulação dos dados
import pandas as pd
import numpy as np

# Visualização dos dados
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

print("Ferramentas importadas com sucesso!")

Ferramentas importadas com sucesso!


In [6]:
# Lendo a base de pedidos da Olist
df_orders = pd.read_csv('../data/olist_orders_dataset.csv')

# Exibir as 3 primeiras linhas
df_orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


### Entendendo a Estrutura dos Dados

Para responder à pergunta de negócio, precisarei relacionar as bases da Olist, começando por:

**1. Tabela de Pedidos (`df_orders`):**
Ela é o "cabeçalho" da compra. Embora não contenha os valores financeiros, ela nos fornece duas informações vitais para a integridade da nossa análise:
* **`order_status`:** Essencial para garantir que só contaremos o faturamento de pedidos reais (ex: status `delivered`), descartando compras canceladas.
* **`order_purchase_timestamp`:** O carimbo de data. Sem ele, seria impossível analisarmos o crescimento da Olist ao longo do tempo (meses/anos).

In [7]:
# 1. Carregando a tabela de itens (onde estão os preços e fretes)
df_items = pd.read_csv('../data/olist_order_items_dataset.csv')

# 2. Descobrindo o tamanho da tabela (quantas linhas e colunas ela tem)
print(f"O tamanho da tabela de itens é: {df_items.shape}")

# 3. Exibindo apenas as 3 primeiras linhas para conhecermos os dados
df_items.head(3)

O tamanho da tabela de itens é: (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


### Entendendo as Variáveis: Itens (Faturamento e Frete)
Nesta segunda tabela (`df_items`), encontramos os indicadores para responder à nossa pergunta principal de negócios:

* **`price` (Preço do Produto):** Classificada como **Variável Quantitativa Contínua**. 
  * *Justificativa Estatística:* É "Quantitativa" pois é um número real que nos permite fazer operações matemáticas (como somar o faturamento total ou calcular a média). É "Contínua" porque o valor financeiro aceita quebras e frações (casas decimais, como R$ 29,90). Representa a base do nosso Faturamento.

* **`freight_value` (Valor do Frete):** Classificada como **Variável Quantitativa Contínua**. 
  * *Justificativa Estatística:* Segue a mesma regra matemática do preço, sendo um valor financeiro que aceita frações decimais. Representa o custo logístico exato cobrado do cliente para aquela entrega.